# Main Figure Revision

Notebook orchestration for the manuscript figures. Shared computation and plotting helpers live in `data_synthesis/src/main_figures_revision.py` so diffs are traceable and notebook JSON stays small.


In [ ]:
from pathlib import Path
import importlib
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]

pkg_root = repo_root / "data_synthesis"
if str(pkg_root) not in sys.path:
    sys.path.insert(0, str(pkg_root))

import src.main_figures_revision as figrev
figrev = importlib.reload(figrev)
from IPython.display import display

datasets, summary = figrev.initialize_datasets()
print(
    f"Run mode: {figrev.RUN_MODE} | CVAE_EPOCHS={figrev.CVAE_EPOCHS}, "
    f"AUC_REPEATS={figrev.AUC_REPEATS}, TSTR_REPEATS={figrev.TSTR_REPEATS}, "
    f"ABLATION_REPEATS={figrev.ABLATION_REPEATS}, NOISE_REPEATS={figrev.NOISE_REPEATS}"
)
display(summary)


## Cached Compute Helpers


In [ ]:
display(figrev.cache_status())


## Figure 2. PCA


In [ ]:
auc_runs = figrev.get_auc_runs(force=False)
fig2_6 = figrev.plot_figure2_six_panel(auc_runs)
display(fig2_6)


In [ ]:
metric_table = figrev.get_metric_table(force=False)
main_metric_summary_fig, main_metric_summary_table = figrev.plot_main_metric_summary_table(metric_table)
structure_metrics_for_supp = figrev.compute_figure4_frobenius_summary(datasets)
supp_metric_summary_fig, supp_metric_summary_table = figrev.plot_supplementary_metric_table(metric_table, structure_metrics_for_supp)
display(main_metric_summary_fig)
display(main_metric_summary_table)
display(supp_metric_summary_fig)
display(supp_metric_summary_table)


In [ ]:
supp_metric_summary_table.to_latex(
    "supp_metric_summary_table.tex",
    index=False,
    float_format="%.3f",
    caption="Summary of synthetic data quality metrics across datasets and methods.",
    label="tab:supp_metric_summary",
)


## Figure 4: Synthetic Deviation from Real Conditional-Dependence Structure

Question: do synthetic datasets preserve the same conditional-dependency structure as the real dataset?

Figure 4 uses Graphical Lasso rather than ordinary pairwise correlation. Graphical Lasso estimates a sparse precision matrix, where off-diagonal entries encode conditional relationships among features. Each dataset is plotted with the same edge-status matrix workflow used by the supplemental panels.

`precision-matrix discrepancy = ||Theta_real - Theta_synthetic||_F`

Main readout:

- **A-D. Synthetic method vs real edge-status matrices**: preserved, real-only/lost, synthetic-only, and absent conditional-dependency edges.
- **E. Structural summary metrics**: all datasets and methods summarized quantitatively. Lower Frobenius discrepancy is better.


## Figure 4 Helpers


In [ ]:
figure4_supp_real_data, figure4_supp_synthetic_data, figure4_supp_feature_names = figrev._get_figure4_precision_inputs()


## Figure 4 Edge-Status Reading Example

This restores the explanatory example figure from the older supplemental workflow: a small edge-status matrix window on the left and the selected feature row converted into a graph on the right.

In [ ]:
figure4_supplemental_results, figure4_supplemental_metrics = figrev.build_figure4_supplemental_figures_inline(
    real_data=figure4_supp_real_data,
    synthetic_data=figure4_supp_synthetic_data,
    feature_names=figure4_supp_feature_names,
    seed=figrev.SEED,
)
fig4_edge_status_examples = figure4_supplemental_results["HIV examples"]

display(fig4_edge_status_examples.fig)
display(figure4_supplemental_results["Breast Cancer"].fig)
display(figure4_supplemental_results["Diabetes"].fig)
display(figure4_supplemental_metrics.sort_values(["figure_dataset", "dataset", "method"]))


In [ ]:
fig4_result = figrev.plot_figure4_edge_status("HIV")
fig4_supp_hiv = fig4_result
corr_summary = fig4_result.metrics

fig4_tsne_hiv = fig4_result
fig4_tsne_group_summaries = figrev.pd.concat([
    fig4_result.preserve_group_summary.assign(dataset="HIV", panel="preserve"),
    fig4_result.lost_group_summary.assign(dataset="HIV", panel="lost"),
    fig4_result.synthetic_only_group_summary.assign(dataset="HIV", panel="synthetic-only"),
], ignore_index=True)

display(fig4_result.fig)
display(corr_summary.sort_values(["dataset", "method"]))
display(fig4_result.feature_index)
display(fig4_tsne_group_summaries[["dataset", "panel", "cluster_id", "method", "n_features", "center_x", "center_y"]])


### Figure 4 Panel E: Global Structural Deviation


In [ ]:
fig4_panel_e = figrev.plot_figure4_metric_summary(
    corr_summary,
    dataset_order=figrev.DATASET_ORDER,
    method_order=figrev.METHOD_ORDER,
)
display(fig4_panel_e)


## Figure 4 Supplementary Edge-Status Matrices

In [ ]:
# Breast Cancer and Diabetes Figure 4 panels are generated only by
# data_synthesis/src/make_figure4_supplementals.py.


## Figure 5: Noise Sensitivity

In [ ]:
noise_df = figrev.get_noise_sensitivity(force=False)
fig5_noise, noise_auc_floor = figrev.plot_figure5_noise(noise_df)
display(fig5_noise)
display(noise_auc_floor)


## Figure 6: Single Combined Reverse-Ablation Figure

This uses the all-dataset A-C function added to the outline notebook, and keeps the x-axis as percentage of top discriminator-ranked features removed so datasets with different numbers of features are comparable.


In [ ]:
ablation_df = figrev.get_reverse_ablation(force=False)
fig6_ac = figrev.plot_figure6_ablation_all_datasets(ablation_df)
display(fig6_ac)
